In [8]:
import pandas as pd
import talib
import pandas_ta as ta
initial_df = pd.read_csv("btcusdt_1h.csv")

final_df = initial_df.copy()
initial_df

,datetime,open,high,low,close,volume
0,2018-01-01 05:30:00,13715.65,13715.65,13576.28,13600.00,33.617798
1,2018-01-01 05:35:00,13600.00,13600.00,13501.01,13554.58,40.528679
2,2018-01-01 05:40:00,13554.58,13569.97,13400.01,13556.15,49.469536
3,2018-01-01 05:45:00,13533.75,13547.73,13402.00,13430.52,32.725614
4,2018-01-01 05:50:00,13440.01,13459.99,13410.44,13439.94,26.614135
...,...,...,...,...,...,...
422337,2022-01-12 05:10:00,42794.82,42822.21,42753.01,42790.03,61.098870
422338,2022-01-12 05:15:00,42790.04,42819.89,42718.77,42736.01,39.718990
422339,2022-01-12 05:20:00,42736.01,42736.02,42633.97,42674.32,55.094370
422340,2022-01-12 05:25:00,42673.60,42739.92,42655.74,42729.29,69.675370


In [9]:
atr = talib.ATR(final_df['high'], final_df['low'], final_df['close'], timeperiod=15)

# Assign ATR values to a new column in your DataFrame
final_df['atr'] = atr

In [10]:
from ta.volatility import BollingerBands

# Initialize Bollinger Bands Indicator
indicator_bb = BollingerBands(close=final_df["close"], window=25, window_dev=2.5)
curr_sig=0
closed=True
final_df['bb_bbm_vals'] = indicator_bb.bollinger_mavg()
final_df['bb_bbh_vals'] = indicator_bb.bollinger_hband()
final_df['bb_bbl_vals'] = indicator_bb.bollinger_lband()

In [11]:
from ta.utils import dropna
from ta.volume import OnBalanceVolumeIndicator

# Initialize On Balance Volume Indicator
indicator_obv = OnBalanceVolumeIndicator(close=final_df["close"],volume=final_df["volume"])
final_df['obv_values'] = indicator_obv.on_balance_volume()
#ema = exponential moving average
final_df["obv_ema"] = final_df['obv_values'].ewm(span=200, adjust=False).mean()

In [12]:
final_df["ema_25"]=final_df["close"].ewm(span=25, adjust=False).mean()
final_df["ema_50"]=final_df["close"].ewm(span=40, adjust=False).mean()
final_df["ema_70"]=final_df["close"].ewm(span=60, adjust=False).mean()

#condition to enter long trade
def strat_long_entry(final_df,bar):

    if final_df["ema_25"].iloc[bar]>final_df["ema_50"].iloc[bar] :
        return True
    else:
        return False

# condition to enter short trade
def strat_short_entry(final_df,bar):
    if final_df["ema_25"].iloc[bar]<final_df["ema_50"].iloc[bar] :
        return True
    else:
        return False

#condition to exit long trade
def strat_long_exit(final_df,bar):

    if final_df["ema_25"].iloc[bar]<final_df["ema_70"].iloc[bar] and final_df["ema_25"].iloc[bar-1]>=final_df["ema_70"].iloc[bar-1]:
        return True
    else:
        return False

#condition to exit short trade
def strat_short_exit(final_df,bar):
    if final_df["ema_25"].iloc[bar]>final_df["ema_70"].iloc[bar] and final_df["ema_25"].iloc[bar-1]<=final_df["ema_70"].iloc[bar-1]:
        return True
    else:
        return False

#stop loss conditions for long and short trades
def long_stop_loss(bar,long_entry_price):
    if long_entry_price==None:
        return False
    elif final_df["close"].iloc[bar]<long_entry_price-2.6*final_df["atr"].iloc[bar]\
        and final_df["bb_bbl_vals"].iloc[bar]>final_df["close"].iloc[bar]\
        and final_df["obv_ema"].iloc[bar]>final_df["obv_values"].iloc[bar]:
        return True
    else:
        return False
    

def short_stop_loss(bar,short_entry_price):
    
    if short_entry_price==None:
        return False
    elif final_df["close"].iloc[bar]>short_entry_price+2.6*final_df["atr"].iloc[bar]\
    and final_df["bb_bbh_vals"].iloc[bar]<final_df["close"].iloc[bar]\
    and final_df["obv_values"].iloc[bar]>final_df["obv_ema"].iloc[bar]:
        return True
    else:
        return False

In [13]:

signals_final=[0,0]
temp=[]
curr_sig=0
closed=True
import statistics
from statistics import mode
    #### set conditions for long_entry and long_exit
print(len(final_df))
long_entry_price=None
short_entry_price=None
for i in range(2,len(final_df)):
    if final_df.iloc[i].isna().any():
        signals_final.append(0)

    else:
        dic={}
        long_entry=strat_long_entry(final_df,i)

        short_exit=strat_short_exit(final_df,i)
        short_entry=strat_short_entry(final_df,i)
        long_exit=strat_long_exit(final_df,i)
        #stop loss ocnditions
        con1=short_stop_loss(i,short_entry_price)
        con2=long_stop_loss(i,long_entry_price)


        if long_entry or short_exit or con1:
              #go long
              if long_entry and closed==True:
                  #previous trade closed, can open new one
                  signals_final.append(1)
                  long_entry_price=final_df["close"].iloc[i]
                  closed=False
                  curr_sig=1
              else:
                                       
                  if ((con1 or short_exit)and curr_sig==-1):
                      closed=True
                      curr_sig=0
                      signals_final.append(1)
                      short_entry_price=None
                  else:
                      signals_final.append(0)
          # set -1
        elif short_entry or long_exit or con2:
              if short_entry and closed==True:
                  signals_final.append(-1)
                  short_entry_price=final_df["close"].iloc[i]
                  closed=False
                  curr_sig=-1
              else:
                  
                  if (long_exit or con2 ) and curr_sig==1:
                      
                      closed=True
                      curr_sig=0
                      signals_final.append(-1)
                      long_entry_price=None
                  else:
                      signals_final.append(0)
        else:
            signals_final.append(0)


422342


In [14]:
final_signals_values=pd.DataFrame(signals_final)
initial_df["signals"]=signals_final

initial_df.to_csv("output_new.csv")